# ETHUSDT Quantitative Research Platform — Notebook 01
## Interactive Strategy Research, Market Intelligence & Backtesting

This notebook provides interactive quantitative research modes without requiring any modification to underlying Python source code.

### Supported Modes:
- `DATA_AUDIT`: Inspect canonical dataset partitions, manifests, SHA-256 fingerprints, and continuity.
- `FEATURE_VALIDATION`: Compute and visually inspect Swings, BOS/CHoCH, FVGs, Liquidity Sweeps, and Regimes.
- `BASELINE`: Execute benchmark baseline strategies (EMA Trend, RSI Reversion, Breakout, Random).
- `STRATEGY_RESEARCH`: Execute and analyze advanced SMC, Structure, Sweep, FVG, or Regime strategies.
- `EXPERIMENT_COMPARE`: Compare multiple experiments from the permanent Research Ledger.

In [ ]:
# =============================================================================
# 1. RESEARCH CONFIGURATION & PARAMETERS
# =============================================================================
MODE = "STRATEGY_RESEARCH"  # Options: 'DATA_AUDIT', 'FEATURE_VALIDATION', 'BASELINE', 'STRATEGY_RESEARCH', 'EXPERIMENT_COMPARE'

SYMBOL = "ETHUSDT"
TIMEFRAME = "15m"
START_DATE = None  # Optional 'YYYY-MM-DD'
END_DATE = None    # Optional 'YYYY-MM-DD'

# Strategy Selection (Options from StrategyCatalog.list_strategies()):
# 'liquidity_sweep_fvg', 'structure_continuation', 'liquidity_sweep_reversal', 
# 'fvg_continuation', 'regime_mean_reversion', 'ema_trend', 'rsi_reversion'
STRATEGY = "liquidity_sweep_fvg"

INITIAL_CAPITAL = 10000.0
RISK_PER_TRADE = 0.01  # 1% per trade
MIN_RR = 1.5
REGISTER_EXPERIMENT = True
SHOW_TRADE_LOG = True

In [ ]:
# =============================================================================
# 2. ENVIRONMENT INITIALIZATION
# =============================================================================
import sys
from pathlib import Path

# Add repository root and src to path
for p in [Path.cwd(), Path.cwd().parent, Path.cwd() / "src"]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import polars as pl
from quant_platform.config.settings import settings
from quant_platform.data.storage.canonical import CanonicalStorage
from quant_platform.data.timeframes.resampler import CausalResampler
from quant_platform.strategies.catalog import StrategyCatalog
from quant_platform.backtest.engine import BacktestEngine
from quant_platform.backtest.costs import CostModel
from quant_platform.research.ledger import ResearchLedger
from quant_platform.research.reporting import ReportGenerator
from quant_platform.features.catalog import FeatureCatalog
from quant_platform.features.structure.market_structure import MarketStructureEngine
from quant_platform.features.smc.fvg import FvgEngine
from quant_platform.features.smc.liquidity import LiquidityEngine
from quant_platform.regimes.engine import MarketRegimeEngine

settings.ensure_directories()
storage = CanonicalStorage()
df_1m = storage.read_range(symbol=SYMBOL)
print(f"Loaded Canonical 1m Dataset: {len(df_1m):,} bars")
if not df_1m.is_empty():
    df_tf = CausalResampler.resample(df_1m, target_timeframe=TIMEFRAME)
    print(f"Resampled to {TIMEFRAME}: {len(df_tf):,} bars (Causally aligned)")

In [ ]:
# =============================================================================
# 3. EXECUTE SELECTED RESEARCH MODE
# =============================================================================
if MODE == "DATA_AUDIT":
    from quant_platform.data.manifest.manifest_manager import DatasetManifestManager
    from quant_platform.data.validation.integrity import DataIntegrityValidator
    
    manifest_mgr = DatasetManifestManager(storage=storage)
    manifest = manifest_mgr.generate_manifest(symbol=SYMBOL, timeframe="1m")
    validator = DataIntegrityValidator()
    rep = validator.validate_dataset(df_1m, timeframe="1m")
    
    print(f"--- DATASET AUDIT REPORT ---")
    print(f"Symbol: {manifest.symbol} | Timeframe: {manifest.timeframe}")
    print(f"SHA-256 Fingerprint: {manifest.canonical_fingerprint}")
    print(f"Total Bars: {manifest.total_rows:,}")
    print(f"Integrity Status: {'VALID' if rep.is_valid else 'INVALID'}")
    print(f"Gaps Found: {len(rep.gaps)} | Duplicates: {len(rep.duplicate_indices)}")

elif MODE == "FEATURE_VALIDATION":
    print("Computing causal market intelligence and SMC features...")
    df_feat = MarketStructureEngine.compute_market_structure(df_tf)
    df_feat, fvgs = FvgEngine.detect_and_track_fvgs(df_feat)
    df_feat, sweeps = LiquidityEngine.detect_liquidity_sweeps(df_feat)
    df_feat = MarketRegimeEngine.classify_regimes(df_feat)
    
    print(f"--- FEATURE AUDIT SUMMARY ---")
    print(f"Total FVGs detected: {len(fvgs):,}")
    print(f"Total Liquidity Sweeps: {len(sweeps):,}")
    print(f"Bullish BOS (Close): {df_feat['is_bos_close_bullish'].sum()}")
    print(f"Bearish BOS (Close): {df_feat['is_bos_close_bearish'].sum()}")
    print(f"Bullish CHoCH: {df_feat['is_choch_bullish'].sum()}")
    print(f"Bearish CHoCH: {df_feat['is_choch_bearish'].sum()}")
    print("\nRecent 5 bars market structure & regime:")
    display_cols = ["open_time", "close", "structural_trend", "regime_tag", "active_bullish_fvg_count", "is_sellside_reclaim"]
    print(df_feat.select([c for c in display_cols if c in df_feat.columns]).tail(5))

elif MODE in ["BASELINE", "STRATEGY_RESEARCH"]:
    strat_cls = StrategyCatalog.get_strategy_class(STRATEGY)
    if not strat_cls:
        raise ValueError(f"Strategy '{STRATEGY}' not found in StrategyCatalog. Available: {StrategyCatalog.list_strategies()}")
    
    strategy = strat_cls()
    print(f"Executing Backtest: {strategy.metadata.strategy_id} (v{strategy.metadata.version})")
    print(f"Hypothesis: {strategy.metadata.hypothesis}")
    
    cost_model = CostModel(maker_fee_rate=0.0002, taker_fee_rate=0.0005, slippage_bps=2.0)
    engine = BacktestEngine(cost_model=cost_model, initial_capital=INITIAL_CAPITAL, risk_per_trade_fraction=RISK_PER_TRADE)
    result = engine.run(df_tf, strategy)
    m = result.metrics
    
    print(f"\n================ PERFORMANCE METRICS ================")
    print(f"Net Return:        {m.total_net_return:+.2f}%")
    print(f"Trade Count:       {m.trade_count}")
    print(f"Win Rate:          {m.win_rate:.1f}%")
    print(f"Profit Factor:     {m.profit_factor:.2f}")
    print(f"Sharpe Ratio:      {m.sharpe_ratio:.2f}")
    print(f"Max Drawdown:      {m.max_drawdown_pct:.2f}%")
    print(f"Average R:         {m.average_r:+.2f}R")
    print(f"Expectancy:        ${m.expectancy:.2f}")
    print(f"Intrabar Ambiguities: {m.intrabar_ambiguity_count}")
    print(f"=====================================================\n")
    
    if REGISTER_EXPERIMENT:
        ledger = ResearchLedger()
        exp_id = ledger.generate_experiment_id()
        from quant_platform.domain.experiment import ExperimentRecord, ExperimentStatus
        from datetime import datetime, timezone
        
        first_ts = int(df_tf['open_time'].min())
        last_ts = int(df_tf['open_time'].max())
        rec = ExperimentRecord(
            experiment_id=exp_id,
            hypothesis=strategy.metadata.hypothesis,
            strategy_id=strategy.metadata.strategy_id,
            strategy_version=strategy.metadata.version,
            feature_set_version="v2",
            parameters=strategy.metadata.parameters,
            parameter_hash=ledger._compute_parameter_hash(strategy.metadata.parameters),
            dataset_fingerprint="canonical_ETHUSDT",
            symbol=SYMBOL,
            timeframe=TIMEFRAME,
            date_range_start=datetime.fromtimestamp(first_ts/1000.0, tz=timezone.utc).strftime("%Y-%m-%d"),
            date_range_end=datetime.fromtimestamp(last_ts/1000.0, tz=timezone.utc).strftime("%Y-%m-%d"),
            cost_model=cost_model.model_dump(),
            execution_model="EVENT_AWARE_TAKER_ENTRY_LIMIT_TP_MARKET_SL",
            risk_model="STRUCTURAL_ATR_FIXED_RISK_1PCT",
            metrics=m,
            trade_count=m.trade_count,
            status=ExperimentStatus.COMPLETED if m.total_net_return > 0 else ExperimentStatus.REJECTED,
            conclusion=f"{strategy.metadata.strategy_id} executed with {m.trade_count} trades and {m.total_net_return:+.2f}% net return.",
        )
        ledger.register_experiment(rec)
        reporter = ReportGenerator()
        html_path = reporter.generate_html_report(rec, result.ledger)
        print(f"Experiment Registered in Permanent Ledger: {exp_id}")
        print(f"HTML Report Generated: {html_path}")

elif MODE == "EXPERIMENT_COMPARE":
    ledger = ResearchLedger()
    exps = ledger.list_experiments()
    print(f"Total Experiments in Permanent Research Ledger: {len(exps)}")
    for e in exps:
        print(f" - {e['experiment_id']}: {e['strategy_id']} | Status: {e['status']} | Net Ret: {e.get('net_return', 0.0):+.2f}% | Win Rate: {e.get('win_rate', 0.0):.1f}%")